In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Real-world Palmer Penguins dataset (344 penguins), hosted on GitHub
url = "https://raw.githubusercontent.com/allisonhorst/palmerpenguins/master/inst/extdata/penguins.csv"
df = pd.read_csv(url)

print("Raw shape:", df.shape)
df[["species", "island", "body_mass_g", "bill_length_mm", "sex"]].head()

Raw shape: (344, 8)


,species,island,body_mass_g,bill_length_mm,sex
0,Adelie,Torgersen,3750.0,39.1,male
1,Adelie,Torgersen,3800.0,39.5,female
2,Adelie,Torgersen,3250.0,40.3,female
3,Adelie,Torgersen,NaN,NaN,NaN
4,Adelie,Torgersen,3450.0,36.7,female


In [2]:
# Drop the few rows with missing body_mass_g (needed for every method below)
df = df.dropna(subset=["body_mass_g", "species", "island"]).reset_index(drop=True)
N = len(df)
print("Population size after cleaning:", N)

print("\nSpecies distribution (will be our STRATA):")
print(df["species"].value_counts(normalize=True).round(3))

print("\nIsland distribution (will be our CLUSTERS):")
print(df["island"].value_counts(normalize=True).round(3))

Population size after cleaning: 342

Species distribution (will be our STRATA):
Adelie       0.442
Gentoo       0.360
Chinstrap    0.199
Name: species, dtype: float64

Island distribution (will be our CLUSTERS):
Biscoe       0.488
Dream        0.363
Torgersen    0.149
Name: island, dtype: float64


In [3]:
TRUE_MEAN_MASS = df["body_mass_g"].mean()
print(f"TRUE population mean body mass: {TRUE_MEAN_MASS:.1f} g")

# Sanity check: body mass clearly differs by species -> a good candidate for stratification
print("\nMean body mass by species:")
print(df.groupby("species")["body_mass_g"].mean().round(1))     #Adelie, Chinstrap, Gentoo.

print("\nMean body mass by island:")                           #Biscoe, Dream, Torgersen
print(df.groupby("island")["body_mass_g"].mean().round(1))

TRUE population mean body mass: 4201.8 g

Mean body mass by species:
species
Adelie       3700.7
Chinstrap    3733.1
Gentoo       5076.0
Name: body_mass_g, dtype: float64

Mean body mass by island:
island
Biscoe       4716.0
Dream        3712.9
Torgersen    3706.4
Name: body_mass_g, dtype: float64


In [4]:
SAMPLE_SIZE = 100

def srs_sample(data, n, seed):
    return data.sample(n=n, random_state=seed)   # seed- a number that controls randomness

srs = srs_sample(df, SAMPLE_SIZE, seed=1)
srs_estimate = srs["body_mass_g"].mean()

print(f"SRS sample size: {len(srs)}")
print(f"SRS estimated mean body mass: {srs_estimate:.1f} g")
print(f"Absolute error vs true value: {abs(srs_estimate - TRUE_MEAN_MASS):.1f} g")

SRS sample size: 100
SRS estimated mean body mass: 4154.0 g
Absolute error vs true value: 47.8 g


In [5]:
def stratified_sample(data, n, strata_col, seed):
    parts = []
    for stratum_value, group in data.groupby(strata_col):
        stratum_n = max(1, round(n * len(group) / len(data))) #For each species, it figures out how many rows to sample from that species so the sample mirrors the population's proportions.
        parts.append(group.sample(n=stratum_n, random_state=seed))
    return pd.concat(parts)

strat = stratified_sample(df, SAMPLE_SIZE, "species", seed=1)
strat_estimate = strat["body_mass_g"].mean()

print(f"Stratified sample size: {len(strat)}")
print("Species proportions in sample vs population:")
comparison = pd.DataFrame({
    "population_%": df["species"].value_counts(normalize=True).round(3),
    "sample_%": strat["species"].value_counts(normalize=True).round(3)
})
print(comparison)

print(f"\nStratified estimated mean body mass: {strat_estimate:.1f} g")
print(f"Absolute error vs true value: {abs(strat_estimate - TRUE_MEAN_MASS):.1f} g")

Stratified sample size: 100
Species proportions in sample vs population:
           population_%  sample_%
Adelie            0.442      0.44
Gentoo            0.360      0.36
Chinstrap         0.199      0.20

Stratified estimated mean body mass: 4247.2 g
Absolute error vs true value: 45.5 g


In [6]:
## 6. Experiment 3 — Cluster Sampling (by Island)

`island` is used to define **clusters** — penguins measured at the same island (Biscoe, Dream, or Torgersen). Instead of picking individuals, we randomly select **whole islands** and take every penguin recorded there.

There are only 3 clusters here (a real limitation of this dataset, same as most real-world cluster variables), so this experiment selects 1 island at a time — a clean illustration of why cluster sampling with *few* clusters can be imprecise, especially since Biscoe island happens to be dominated by the heavier Gentoo species (Section 3).

SyntaxError: invalid syntax (2902703649.py, line 3)

In [7]:
def cluster_sample(data, cluster_col, seed, n_clusters=1):
    rng = np.random.default_rng(seed)
    clusters = data[cluster_col].unique()  # u get names of cluster
    chosen = rng.choice(clusters, size=n_clusters, replace=False)
    return data[data[cluster_col].isin(chosen)], chosen

cluster, chosen_islands = cluster_sample(df, "island", seed=1, n_clusters=1)
cluster_estimate = cluster["body_mass_g"].mean()

print(f"Island(s) selected: {list(chosen_islands)}")
print(f"Cluster sample size: {len(cluster)}")
print(f"Cluster estimated mean body mass: {cluster_estimate:.1f} g")
print(f"Absolute error vs true value: {abs(cluster_estimate - TRUE_MEAN_MASS):.1f} g")

Island(s) selected: ['Biscoe']
Cluster sample size: 167
Cluster estimated mean body mass: 4716.0 g
Absolute error vs true value: 514.3 g


In [8]:
results = pd.DataFrame({
    "Method": ["SRS", "Stratified (species)", "Cluster (island)"],
    "Sample Size": [len(srs), len(strat), len(cluster)],
    "Estimated Mean Mass (g)": [srs_estimate, strat_estimate, cluster_estimate],
    "Absolute Error (g)": [abs(srs_estimate - TRUE_MEAN_MASS),
                            abs(strat_estimate - TRUE_MEAN_MASS),
                            abs(cluster_estimate - TRUE_MEAN_MASS)]
})
results.round(1)

,Method,Sample Size,Estimated Mean Mass (g),Absolute Error (g)
0,SRS,100,4154.0,47.8
1,Stratified (species),100,4247.2,45.5
2,Cluster (island),167,4716.0,514.3
